# Turn your bank statement into a budget: Apertus 1.5B, LoRA with MLX, on a Mac

A bank statement is a list of messy lines like `SQ *BLUE BOTTLE 0423 SAN FRANCISCO CA`. A budget needs one category
per line: Restaurants, Rent, Income, ... We teach Swiss AI's **Apertus v1.1 1.5B Instruct** to file every line into
one of 17 budget categories, with **LoRA** in **mlx-lm** on the Mac's GPU, then add up a month into a budget.

In [ ]:
import json

import pandas as pd
from huggingface_hub import hf_hub_download
from mlx_lm import load

from budget import (BASE, CATS, DATASET, SYSTEM, budget, classify, evaluate, free, hold_out, load_statement,
                    merchant, merchant_groups, messages, take, train_usage)

## 0. The machine

The same MacBook Pro as every lesson in this series. MLX runs on the Mac's GPU.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

## 1. The data

[`DoDataThings/us-bank-transaction-categories-v2`](https://huggingface.co/datasets/DoDataThings/us-bank-transaction-categories-v2)
(MIT): 68,000 US bank-statement lines in 17 categories. The lines are **synthetic**: generated from real merchant
names in the formats of 8 banks and payment apps. Each line starts with `[debit]` (money out) or `[credit]` (money in).

In [ ]:
path = hf_hub_download(DATASET, "transactions-synthetic.csv", repo_type="dataset")
d = pd.read_csv(path)
print(len(d), "lines,", d.category.nunique(), "categories")
d.sample(8, random_state=7)

### Trap 1: copies

Many lines appear more than once. If the same line is in training and in the test, the test only checks memory.
We keep each line once, and drop the few lines that carry two different categories.

In [ ]:
print(d.description.nunique(), "different lines")
two = d.groupby("description").category.nunique()
two = two[two > 1].index
d = d[~d.description.isin(two)].drop_duplicates("description").reset_index(drop=True)
len(two), len(d)

### Trap 2: the same shop on both sides

Even without copies, a random split puts the same shops in training and in the test. A real statement has shops
the model has never seen. So we **test on shops it has never seen**: strip the bank's wrapper (`SQ *`, `TST*`,
`PYPL *`, store numbers, addresses) to get the shop, group the lines by shop, and hold whole shops out.

In [ ]:
lines = ["[debit] SQ *BLUE BOTTLE 0423 SAN FRANCISCO CA", "[debit] TST*GOLDEN DRAGON - OAKLAND", "[debit] wholefds #8860 chicago",
         "[debit] WHOLE FOODS #82216"]
for line, group in zip(lines, merchant_groups(lines)):
    print(f"{line:<46} -> {merchant(line):<14} group {group}")

In [ ]:
d["group"] = merchant_groups(d.description)
held = hold_out(d)
print(d.group.nunique(), "shop groups,", len(held), "held out")
new_shops, seen_shops = d[d.group.isin(held)], d[~d.group.isin(held)]
test_new = take(new_shops, 100, seed=1)      # 100 lines per category from shops never seen in training
test_known = take(seen_shops, 100, seed=2)   # 100 per category from known shops (new lines, same shops)
rest = seen_shops.drop(test_known.index)
valid = take(rest, 50, seed=3)
train = rest.drop(valid.index)
print(f"train {len(train):,} | valid {len(valid):,} | test, never-seen shops {len(test_new):,} | test, known shops {len(test_known):,}")
print("never-seen shop groups that are also in train:", test_new.group.isin(set(train.group)).sum())

Every example becomes a short chat: the system prompt with the 17 categories, the bank line, and the category.
`mlx_lm.lora` trains on files of these, one JSON object per line.

In [ ]:
print(SYSTEM)
messages(train.description.iloc[0], train.category.iloc[0])

In [ ]:
import os
os.makedirs("data", exist_ok=True)
for name, df in [("train", train.sample(frac=1, random_state=4)), ("valid", valid)]:
    with open(f"data/{name}.jsonl", "w") as f:
        f.writelines(json.dumps({"messages": messages(x, c)}) + "\n" for x, c in zip(df.description, df.category))
!wc -l data/*.jsonl

## 2. Before: the base model

Apertus v1.1 1.5B Instruct, from Swiss AI (ETH Zurich and EPFL), Apache-2.0. Its model card also asks users to
follow the Apertus Acceptable Use Policy. Same system prompt, one line in, one category out.

In [ ]:
model, tok = load(BASE)
classify(model, tok, ["[debit] SQ *BLUE BOTTLE 0423 SAN FRANCISCO CA"], progress=False)[0]

In [ ]:
base_new = evaluate(model, tok, test_new, "never-seen shops")
base_known = evaluate(model, tok, test_known, "known shops")

### A month of bank lines

`statement.csv` is September 2026 for one person: 31 lines that **we wrote ourselves; every merchant is made up**
(none of them is in the dataset). The base model files them, and we add up the budget.

In [ ]:
st = load_statement()
base_month, _ = classify(model, tok, st.description, progress=False)
budget(st, {"base": base_month})

In [ ]:
del model
free()

## 3. Fine-tune with LoRA

LoRA keeps the model's own weights **frozen** and trains small extra matrices next to its layers: here in all 16
blocks, rank 8. `mask_prompt` computes the loss only on the category, not on the prompt. 1,000 steps of 16 lines =
16,000 lines, about 45 % of the training set. This run needs about 16 GB of memory.

`/usr/bin/time -l` is the Mac's built-in timer: it saves how long training took and its peak memory.

In [ ]:
%%writefile train.yaml
model: swiss-ai/Apertus-v1.1-1.5B-Instruct
train: true
data: data
fine_tune_type: lora
num_layers: 16
lora_parameters: {rank: 8, scale: 20.0, dropout: 0.0}
optimizer: adamw
mask_prompt: true
batch_size: 16
iters: 1000
learning_rate: 1.0e-4
lr_schedule:
  name: cosine_decay
  warmup: 20
  arguments: [1.0e-4, 1000, 1.0e-6]
steps_per_report: 10
steps_per_eval: 500
val_batches: -1
save_every: 1000
max_seq_length: 512
seed: 0
adapter_path: runs/adapters

In [ ]:
!/usr/bin/time -l -o train_usage.txt mlx_lm.lora --config train.yaml

In [ ]:
train_secs, train_peak = train_usage()

## 4. After

Fuse the adapter into the weights: one normal model folder, as fast as the original, and the folder we turn into
an Ollama model later.

In [ ]:
!mlx_lm.fuse --model swiss-ai/Apertus-v1.1-1.5B-Instruct --adapter-path runs/adapters --save-path runs/apertus-bank-budget

In [ ]:
model, tok = load("runs/apertus-bank-budget")
tuned_new = evaluate(model, tok, test_new, "never-seen shops")
tuned_known = evaluate(model, tok, test_known, "known shops")

In [ ]:
ex = ['[debit] AMZN DIGITAL*JDUMU07J9K', '[debit] skillshare*ny2tqmqzwn63', '[credit] AIMCO LEASE PAYMENT', '[debit] ARCO 755801', "[debit] CASHIER'S CHECK #894586", '[debit] PUBLIX #4309']
for x in ex:
    b, t = base_new.set_index("description").loc[x], tuned_new.set_index("description").loc[x]
    print(f"{x:<34} true {t.category:<15} base {b.pred:<15} tuned {t.pred}")

In [ ]:
tuned_month, ms = classify(model, tok, st.description, progress=False)
budget(st, {"base": base_month, "tuned": tuned_month})

In [ ]:
acc = lambda r: 100 * (r.pred == r.category).mean()
print(f"MacBook Pro M5 Max, 36 GB | Apertus v1.1 1.5B, LoRA with mlx-lm | "
      f"{int(train_secs // 60)} min {train_secs % 60:.0f} s, peak {train_peak:.1f} GB")
print(f"never-seen shops: base {acc(base_new):.1f} %  ->  tuned {acc(tuned_new):.1f} %")
print(f"known shops:      base {acc(base_known):.1f} %  ->  tuned {acc(tuned_known):.1f} %")
print(f"the month:        base {sum(p == t for p, t in zip(base_month, st.category))} / {len(st)}  ->  "
      f"tuned {sum(p == t for p, t in zip(tuned_month, st.category))} / {len(st)}")

In [ ]:
del model
free()

## 5. Run it in Ollama

Turn the fused model into a GGUF file with llama.cpp's converter and keep it at 8 bits (Q8_0). We use the llama.cpp
that Unsloth Studio installed in lesson 4 (its Python has the converter's packages). One small fix first: mlx-lm
stores two values of Apertus' activation in a slightly different shape than the converter expects.

In [ ]:
import sys
from pathlib import Path

from budget import fix_for_llama_cpp, modelfile

LLAMA_CPP = Path.home() / ".unsloth/llama.cpp"
STUDIO_PYTHON = Path.home() / ".unsloth/studio/unsloth_studio/bin/python"
fix_for_llama_cpp("runs/apertus-bank-budget", "runs/apertus-bank-budget-hf")
!{STUDIO_PYTHON} {LLAMA_CPP}/convert_hf_to_gguf.py runs/apertus-bank-budget-hf --outfile runs/bank-budget-bf16.gguf --outtype bf16 > runs/convert.log 2>&1; tail -1 runs/convert.log
!{LLAMA_CPP}/llama-quantize runs/bank-budget-bf16.gguf runs/bank-budget-Q8_0.gguf Q8_0 > runs/quantize.log 2>&1; tail -2 runs/quantize.log
!ls -lh runs/*.gguf

The Modelfile gives Ollama the same chat layout and system prompt we trained with.

In [ ]:
open("runs/Modelfile", "w").write(modelfile("bank-budget-Q8_0.gguf"))
print(open("runs/Modelfile").read())
!cd runs && ollama create bank-budget -f Modelfile

In [ ]:
from budget import evaluate_ollama

ollama_new = evaluate_ollama("bank-budget", test_new, "never-seen shops")

## 6. Put it on Hugging Face

The LoRA adapter (for MLX) and the Q8_0 file with its Modelfile (for Ollama) go to a Hugging Face model repo.
Log in once with `hf auth login`.

In [ ]:
from huggingface_hub import HfApi

REPO = "kgptalkie/apertus-1.5b-bank-budget"
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path="runs/adapters", repo_id=REPO, allow_patterns=["adapters.safetensors", "adapter_config.json"])
api.upload_file(path_or_fileobj="runs/bank-budget-Q8_0.gguf", path_in_repo="bank-budget-Q8_0.gguf", repo_id=REPO)
api.upload_file(path_or_fileobj="runs/Modelfile", path_in_repo="Modelfile", repo_id=REPO)
api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO)
print(f"https://huggingface.co/{REPO}")